<a href="https://colab.research.google.com/github/bhushan1729/olfaction-inspired-ner/blob/main/notebooks/hyperparameter_tuning_conll2003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Comprehensive Hyperparameter Tuning
**Dataset**: CoNLL-2003

**Experiments**: 8 systematic variations

1. Baseline (ReLU)
2. Baseline (GELU)
3. More Receptors (256)
4. More Glomeruli (64)
5. Larger LSTM (512)
6. Lower Dropout (0.2)
7. Larger Batch (64)
8. Strong Regularization

**Each experiment auto**:
- Trains & evaluates
- Saves model → Google Drive
- Saves results → GitHub
- Generates curves & plots

## Setup

In [2]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4


In [5]:
# Clone repository
import getpass

# Get your token securely (won't be visible in output)
token = getpass.getpass('Enter your GitHub Personal Access Token: ')

# Clone using token
!git clone https://{token}@github.com/bhushan1729/olfaction-inspired-ner.git
%cd olfaction-inspired-ner


Enter your GitHub Personal Access Token: ··········
Cloning into 'olfaction-inspired-ner'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 182 (delta 82), reused 142 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 1.15 MiB | 3.68 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/olfaction-inspired-ner


In [3]:
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import os
if not os.path.exists('./data/glove.6B.300d.txt'):
    !mkdir -p data
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip
    !unzip -q data/glove.6B.zip -d data/
    !rm data/glove.6B.zip

## Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

model_dir = '/content/drive/MyDrive/olfaction_ner/models'
!mkdir -p {model_dir}

Mounted at /content/drive


## Helpers

In [8]:
import json
import shutil
import matplotlib.pyplot as plt
from pathlib import Path

!mkdir -p comparison_plots experiment_results/CoNLL-2003

all_experiments = []

def save_to_drive(name, save_dir):
    dst = f'/content/drive/MyDrive/olfaction_ner/models/{name}.pt'
    shutil.copy(f'{save_dir}/best_model.pt', dst)
    print(f'✓ Drive: {dst}')

def save_results(name, cfg, res, sd):
    ed = f'experiment_results/CoNLL-2003/{name}'
    Path(ed).mkdir(parents=True, exist_ok=True)

    with open(f'{ed}/metadata.json', 'w') as f:
        json.dump({'name': name, 'config': cfg}, f, indent=2)

    with open(f'{ed}/results.json', 'w') as f:
        json.dump(res, f, indent=2)

    all_experiments.append({'name': name, 'config': cfg, 'results': res})

def plot_curves(name, res, path):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    epochs = [e['epoch'] for e in res['epochs']]

    ax[0].plot(epochs, [e['train']['total_loss'] for e in res['epochs']])
    ax[0].set_title(f'{name}: Loss')

    ax[1].plot(epochs, [e['valid']['f1'] for e in res['epochs']], color='green')
    ax[1].set_title(f'{name}: F1')

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()

print('✓ Ready')

✓ Ready


# Experiments

## Experiment 1: Baseline (ReLU)

In [9]:
!python src/train.py --config config/experiments.yaml --experiment olfactory_full --save_dir results/tuning/exp1_relu

2026-01-12 17:27:59.067721: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768238879.087053    3408 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768238879.092949    3408 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768238879.107774    3408 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768238879.107798    3408 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768238879.107802    3408 computation_placer.cc:177] computation placer alr

In [10]:
with open('results/tuning/exp1_relu/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'relu',
    'receptors': 128,
    'glomeruli': 32,
    'lstm': 256,
    'dropout': 0.5,
    'batch': 32,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp1_relu', 'results/tuning/exp1_relu')
save_results('exp1_relu', cfg, results, 'results/tuning/exp1_relu')
plot_curves('Baseline (ReLU)', results, 'comparison_plots/exp1_relu.png')

print(f"✅ Exp1: F1={results['test']['f1']:.4f}")

✓ Drive: /content/drive/MyDrive/olfaction_ner/models/exp1_relu.pt
✅ Exp1: F1=0.7107


## Experiment 2: Baseline (GELU)

In [ ]:
!python src/train.py --config config/tuning_experiments.yaml --experiment activation_gelu --save_dir results/tuning/exp2_gelu

In [ ]:
with open('results/tuning/exp2_gelu/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 128,
    'glomeruli': 32,
    'lstm': 256,
    'dropout': 0.5,
    'batch': 32,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp2_gelu', 'results/tuning/exp2_gelu')
save_results('exp2_gelu', cfg, results, 'results/tuning/exp2_gelu')
plot_curves('Baseline (GELU)', results, 'comparison_plots/exp2_gelu.png')

print(f"✅ Exp2: F1={results['test']['f1']:.4f}")

## Experiment 3: More Receptors

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp3_more_receptors --save_dir results/tuning/exp3_rec256

In [ ]:
with open('results/tuning/exp3_rec256/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 256,
    'glomeruli': 64,
    'lstm': 256,
    'dropout': 0.5,
    'batch': 32,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp3_rec256', 'results/tuning/exp3_rec256')
save_results('exp3_rec256', cfg, results, 'results/tuning/exp3_rec256')
plot_curves('More Receptors (256)', results, 'comparison_plots/exp3_rec256.png')

print(f"✅ Exp3: F1={results['test']['f1']:.4f}")

## Experiment 4: More Glomeruli

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp4_more_glomeruli --save_dir results/tuning/exp4_glom64

In [ ]:
with open('results/tuning/exp4_glom64/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 128,
    'glomeruli': 64,
    'lstm': 256,
    'dropout': 0.5,
    'batch': 32,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp4_glom64', 'results/tuning/exp4_glom64')
save_results('exp4_glom64', cfg, results, 'results/tuning/exp4_glom64')
plot_curves('More Glomeruli (64)', results, 'comparison_plots/exp4_glom64.png')

print(f"✅ Exp4: F1={results['test']['f1']:.4f}")

## Experiment 5: Larger LSTM

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp5_larger_lstm --save_dir results/tuning/exp5_lstm512

In [ ]:
with open('results/tuning/exp5_lstm512/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 128,
    'glomeruli': 32,
    'lstm': 512,
    'dropout': 0.5,
    'batch': 32,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp5_lstm512', 'results/tuning/exp5_lstm512')
save_results('exp5_lstm512', cfg, results, 'results/tuning/exp5_lstm512')
plot_curves('Larger LSTM (512)', results, 'comparison_plots/exp5_lstm512.png')

print(f"✅ Exp5: F1={results['test']['f1']:.4f}")

## Experiment 6: Lower Dropout

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp6_lower_dropout --save_dir results/tuning/exp6_drop02

In [ ]:
with open('results/tuning/exp6_drop02/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 128,
    'glomeruli': 32,
    'lstm': 256,
    'dropout': 0.2,
    'batch': 32,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp6_drop02', 'results/tuning/exp6_drop02')
save_results('exp6_drop02', cfg, results, 'results/tuning/exp6_drop02')
plot_curves('Lower Dropout (0.2)', results, 'comparison_plots/exp6_drop02.png')

print(f"✅ Exp6: F1={results['test']['f1']:.4f}")

## Experiment 7: Larger Batch

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp7_larger_batch --save_dir results/tuning/exp7_batch64

In [ ]:
with open('results/tuning/exp7_batch64/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 128,
    'glomeruli': 32,
    'lstm': 256,
    'dropout': 0.5,
    'batch': 64,
    'sparse': 0.001,
    'diverse': 0.01
}

save_to_drive('exp7_batch64', 'results/tuning/exp7_batch64')
save_results('exp7_batch64', cfg, results, 'results/tuning/exp7_batch64')
plot_curves('Larger Batch (64)', results, 'comparison_plots/exp7_batch64.png')

print(f"✅ Exp7: F1={results['test']['f1']:.4f}")

## Experiment 8: Strong Reg

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp8_strong_reg --save_dir results/tuning/exp8_strongreg

In [ ]:
with open('results/tuning/exp8_strongreg/results.json') as f:
    results = json.load(f)

cfg = {
    'activation': 'gelu',
    'receptors': 128,
    'glomeruli': 32,
    'lstm': 256,
    'dropout': 0.5,
    'batch': 32,
    'sparse': 0.1,
    'diverse': 0.1
}

save_to_drive('exp8_strongreg', 'results/tuning/exp8_strongreg')
save_results('exp8_strongreg', cfg, results, 'results/tuning/exp8_strongreg')
plot_curves('Strong Regularization', results, 'comparison_plots/exp8_strongreg.png')

print(f"✅ Exp8: F1={results['test']['f1']:.4f}")

# Comparison

In [ ]:
import pandas as pd

data = []
for e in all_experiments:
    data.append({
        'Name': e['name'],
        'F1': f"{e['results']['test']['f1']:.4f}",
        'Precision': f"{e['results']['test']['precision']:.4f}",
        'Recall': f"{e['results']['test']['recall']:.4f}"
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))
df.to_csv('comparison_plots/results.csv', index=False)

## Push to GitHub

In [11]:
!git config user.name "Colab"
!git config user.email "colab@exp.local"
!git add experiment_results/ comparison_plots/
!git commit -m "Hyperparameter tuning results"
!git push origin main
print('✅ Pushed to GitHub')

[main 675ed40] Hyperparameter tuning results
 3 files changed, 1163 insertions(+)
 create mode 100644 comparison_plots/exp1_relu.png
 create mode 100644 experiment_results/CoNLL-2003/exp1_relu/metadata.json
 create mode 100644 experiment_results/CoNLL-2003/exp1_relu/results.json
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (9/9), 58.83 KiB | 14.71 MiB/s, done.
Total 9 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/bhushan1729/olfaction-inspired-ner.git
   c64eecf..675ed40  main -> main
✅ Pushed to GitHub
